
# **ML Modelling**

In [ ]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/Int_data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/Int_FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/Int_data_after_corr_20260507_1548
Loading FS from: /content/drive/MyDrive/Int_FS_results_20260507_1548


In [ ]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)



================ FEATURE SETS LOADED ================

ANOVA:
Number of features: 15
Features: ['Test Today - peak HR', 'Age', 'RISK  - Risk Type', 'Predicted Risk Level Encoded', 'Test Today - Termination Cause', 'Test Today - METS', 'Exercise Habit - Mode', 'Occupation', 'Risk Factor - Stress', 'Exercise Habit - Duration', 'Risk Factor - ECHO - EF', 'Diagnosis', 'ROM', 'ECG Resting', 'Risk Factor - DM']

Mutual_Info:
Number of features: 15
Features: ['Test Today - peak HR', 'Test Today - METS', 'Functional Activity', 'Age', 'RISK  - Risk Type', 'Living Environment', 'ROM', 'Risk Factor - Smoking', 'Predicted Risk Level Encoded', 'Weekly_Exercise_Duration', 'Walking', 'Gait', 'Risk Factor - Stress', 'Posture', 'Lives With']

RFE_LR:
Number of features: 15
Features: ['Age', 'Test Today - peak HR', 'Predicted Risk Level Encoded', 'RISK  - Risk Type', 'Risk Factor - Stress', 'Gender', 'Risk Factor - DM', 'Test Today - Termination Cause', 'Risk Factor - Family hx', 'Risk Factor - HPL', '

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# **Conventional Models**




In [ ]:
# SVM MODEL

import numpy as np
import random
import os

from collections import Counter

from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# Automatically adjusts to minority class
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# SVM PIPELINE
# ============================================

svm_model = ImbPipeline([

    ("scaler", StandardScaler()),

    ("ros", RandomOverSampler(
        random_state=SEED
    )),

    ("svm", SVC(
        kernel='rbf',
        probability=True,
        random_state=SEED
    ))
])

# ============================================
# HYPERPARAMETERS
# ============================================

param_dist = {

    "svm__C": [
        0.1,
        1,
        10
    ],

    "svm__gamma": [
        "scale",
        "auto"
    ]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== SVM | FEATURE SET: {fs_name} ==========")

    # ----------------------------------------
    # SELECT FEATURES
    # ----------------------------------------

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # ----------------------------------------
    # SAFE n_iter
    # ----------------------------------------

    total_space = int(np.prod([
        len(v) for v in param_dist.values()
    ]))

    n_iter = min(10, total_space)

    # ----------------------------------------
    # RANDOM SEARCH CV
    # ----------------------------------------

    search = RandomizedSearchCV(
        estimator=svm_model,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # ----------------------------------------
    # TRAIN
    # ----------------------------------------

    search.fit(X_train_sel, y_train)

    # ----------------------------------------
    # TEST
    # ----------------------------------------

    y_pred = search.predict(X_test_sel)

    # ----------------------------------------
    # RESULTS
    # ----------------------------------------

    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 4-Fold Stratified CV


========== SVM | FEATURE SET: ANOVA ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 68.25%
Precision (W): 64.46%
Recall (W): 68.25%
F1 (W): 65.81%
Precision (Macro): 33.95%
Recall (Macro): 34.15%
F1 (Macro): 33.75%


========== SVM | FEATURE SET: Mutual_Info ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 57.14%
Precision (W): 62.32%
Recall (W): 57.14%
F1 (W): 59.22%
Precision (Macro): 39.43%
Recall (Macro): 43.83%
F1 (Macro): 40.08%


========== SVM | FEATURE SET: RFE_LR ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 10}
Accuracy: 61.90%
Precision (W): 57.79%
Recall (W): 61.90%
F1 (W): 59.52%
Precision (Macro): 29.55%
Recall (Macro): 30.45%
F1 (Macro): 29.83%


========== SVM | FEATURE SET: RFE_SVM ==========
Best Params: {'svm__gamma': 'scale', 'svm__C': 1}
Accuracy: 71.43%
Precision (W): 70.38%
Recall (W): 71.43%
F1 (W): 70.47%
Precision (Macro): 37.72%
Recall (Macro): 36.53%
F1 (Macro): 36.86%


=

# **Deep learning**







In [ ]:
# LM-BPNN

import numpy as np
import random
import os

from collections import Counter

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================
# FIXED SEED (REPRODUCIBILITY)
# =========================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================
# SAFE STRATIFIED CV
# Automatically adapts to smallest class
# =========================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================
# LM-BPNN MODEL
# =========================================

lm_bpnn = ImbPipeline([

    # Standardization inside CV
    ("scaler", StandardScaler()),

    # Oversampling inside training folds only
    ("ros", RandomOverSampler(
        random_state=SEED
    )),

    # MLP approximation of LM-BPNN
    ("mlp", MLPClassifier(
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=SEED
    ))
])

# =========================================
# HYPERPARAMETER SEARCH SPACE
# =========================================

param_dist = {

    "mlp__hidden_layer_sizes": [
        (128, 64),
        (128, 128),
        (256, 128)
    ],

    "mlp__alpha": [
        0.0001,
        0.001,
        0.01
    ],

    "mlp__learning_rate_init": [
        0.001,
        0.005
    ]
}

# =========================================
# MAIN LOOP
# =========================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== LM-BPNN | FEATURE SET: {fs_name} ==========")

    # -------------------------------------
    # SELECT FEATURES
    # -------------------------------------

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # -------------------------------------
    # PREVENT RANDOMIZEDSEARCH WARNING
    # -------------------------------------

    total_space = int(np.prod([
        len(v) for v in param_dist.values()
    ]))

    n_iter = min(10, total_space)

    # -------------------------------------
    # RANDOM SEARCH
    # -------------------------------------

    search = RandomizedSearchCV(
        estimator=lm_bpnn,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,   # reproducible
        random_state=SEED
    )

    # -------------------------------------
    # TRAIN
    # -------------------------------------

    search.fit(X_train_sel, y_train)

    # -------------------------------------
    # TEST PREDICTION
    # -------------------------------------

    y_pred = search.predict(X_test_sel)

    # -------------------------------------
    # RESULTS
    # -------------------------------------

    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 4-Fold Stratified CV


========== LM-BPNN | FEATURE SET: ANOVA ==========
Best Params: {'mlp__learning_rate_init': 0.005, 'mlp__hidden_layer_sizes': (128, 128), 'mlp__alpha': 0.01}
Accuracy: 63.49%
Precision (W): 62.55%
Recall (W): 63.49%
F1 (W): 63.01%
Precision (Macro): 32.54%
Recall (Macro): 33.24%
F1 (Macro): 32.88%


========== LM-BPNN | FEATURE SET: Mutual_Info ==========
Best Params: {'mlp__learning_rate_init': 0.005, 'mlp__hidden_layer_sizes': (128, 64), 'mlp__alpha': 0.0001}
Accuracy: 68.25%
Precision (W): 69.86%
Recall (W): 68.25%
F1 (W): 68.98%
Precision (Macro): 41.78%
Recall (Macro): 42.36%
F1 (Macro): 41.88%


========== LM-BPNN | FEATURE SET: RFE_LR ==========
Best Params: {'mlp__learning_rate_init': 0.005, 'mlp__hidden_layer_sizes': (128, 128), 'mlp__alpha': 0.0001}
Accuracy: 68.25%
Precision (W): 64.08%
Recall (W): 68.25%
F1 (W): 66.10%
Precision (Macro): 32.88%
Recall (Macro): 35.21%
F1 (Macro): 34.00%


========== LM-BPNN | FEATURE SET: RFE_SVM ==========
Best 

# **Transformer models**





In [ ]:
# SUPER ENSEMBLE
# RF + TT + TN STACKING

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# BASE MODELS
# =========================================================

def build_models():

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=1
    )

    tt = ImbPipeline([

        ("ros", RandomOverSampler(random_state=SEED)),

        ("scaler", StandardScaler()),

        ("mlp", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=1200,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=SEED
        ))
    ])

    tn = ImbPipeline([

        ("ros", RandomOverSampler(random_state=SEED)),

        ("scaler", StandardScaler()),

        ("mlp", MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            max_iter=1500,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=SEED
        ))
    ])

    return rf, tt, tn

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== SUPER ENSEMBLE | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    rf, tt, tn = build_models()

    # =====================================================
    # OOF PREDICTIONS
    # =====================================================

    rf_oof = cross_val_predict(
        rf,
        X_train_sel,
        y_train,
        cv=cv,
        method='predict_proba',
        n_jobs=1
    )

    tt_oof = cross_val_predict(
        tt,
        X_train_sel,
        y_train,
        cv=cv,
        method='predict_proba',
        n_jobs=1
    )

    tn_oof = cross_val_predict(
        tn,
        X_train_sel,
        y_train,
        cv=cv,
        method='predict_proba',
        n_jobs=1
    )

    # META TRAIN
    X_meta_train = np.hstack([
        rf_oof,
        tt_oof,
        tn_oof
    ])

    # =====================================================
    # FIT FULL TRAIN
    # =====================================================

    rf.fit(X_train_sel, y_train)
    tt.fit(X_train_sel, y_train)
    tn.fit(X_train_sel, y_train)

    # =====================================================
    # TEST PREDICTIONS
    # =====================================================

    rf_test = rf.predict_proba(X_test_sel)
    tt_test = tt.predict_proba(X_test_sel)
    tn_test = tn.predict_proba(X_test_sel)

    X_meta_test = np.hstack([
        rf_test,
        tt_test,
        tn_test
    ])

    # =====================================================
    # META MODEL
    # =====================================================

    meta = Pipeline([

        ('scaler', StandardScaler()),

        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=SEED
        ))
    ])

    meta.fit(X_meta_train, y_train)

    y_pred = meta.predict(X_meta_test)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 4-Fold Stratified CV


========== SUPER ENSEMBLE | FEATURE SET: ANOVA ==========
Accuracy: 49.21%
Precision (W): 59.79%
Recall (W): 49.21%
F1 (W): 52.64%
Precision (Macro): 33.66%
Recall (Macro): 41.07%
F1 (Macro): 32.69%


========== SUPER ENSEMBLE | FEATURE SET: Mutual_Info ==========
Accuracy: 52.38%
Precision (W): 61.63%
Recall (W): 52.38%
F1 (W): 55.57%
Precision (Macro): 37.95%
Recall (Macro): 57.46%
F1 (Macro): 37.32%


========== SUPER ENSEMBLE | FEATURE SET: RFE_LR ==========
Accuracy: 53.97%
Precision (W): 74.01%
Recall (W): 53.97%
F1 (W): 60.15%
Precision (Macro): 44.48%
Recall (Macro): 67.92%
F1 (Macro): 42.11%


========== SUPER ENSEMBLE | FEATURE SET: RFE_SVM ==========
Accuracy: 49.21%
Precision (W): 65.59%
Recall (W): 49.21%
F1 (W): 54.95%
Precision (Macro): 37.47%
Recall (Macro): 41.07%
F1 (Macro): 34.24%


========== SUPER ENSEMBLE | FEATURE SET: LASSO ==========
Accuracy: 46.03%
Precision (W): 66.78%
Recall (W): 46.03%
F1 (W): 52.14%
Precision (Macro): 37.01%
R